Inverter Division (ELM) - PV Paiva
=
Quick sizing calculations for splitting the 12089 strings of PV Paiva between its 464 inverters (29 PSTs), with two module power bins (735 Wp and 740 Wp, 30 modules per string). It is a copy of `Divisão ELM.ipynb` (725 / 730 Wp) with Paiva's values and an extra search with 25-, 26- and 27-string inverters.

**Workflow overview:**
1. Inverters per PST.
2. Max strings per MPPT / per inverter from the MPPT current limits, and the average strings per inverter.
3. Max modules per string from the inverter max input voltage.
4. How many inverters of each module bin get 26 or 27 strings.
5. Helper: combinations of 26- and 27-string inverters that add up to a target number of strings.
6. Helper: combinations of 25-, 26- and 27-string inverters that add up to a target.

**Input prerequisites:** no files are read or written. Everything is hardcoded in the first cell: string counts, module data per bin (`mod1`, `mod2`), inverter limits (`inv`), and the inverter and PST counts.

Inputs and inverters per PST
=
- `strings_total = 12089`, of which `strings_735 = 9671` use 735 Wp modules; the rest (2418) use 740 Wp. Module quantity = strings × 30.
- `mod1` (735 Wp) and `mod2` (740 Wp): datasheet values (Vmp, Voc, Imp, Isc) of each bin.
- `inv`: inverter limits (max input voltage 1500 V, 6 MPPTs, 106 A max current and 180 A max short circuit per MPPT, 6 inputs per MPPT).
- `inverters / PSTs` gives inverters per PST. If it is not an integer, prints the two neighbours (`ceil` / `floor`). Here 464 / 29 = 16 exactly.

In [2]:
import math
strings_total = 12089
strings_735 = 9671
strings_740 = strings_total - strings_735

mod1 = {
    'Potência': 735,
    'Qtd.': strings_735*30,
    'Strings': strings_735,
    'Vmp': 41.41,
    'Voc': 49.77,
    'Imp': 17.75,
    'Isc': 18.72
}

mod2 = {
    'Potência': 740,
    'Qtd.': strings_740*30,
    'Strings': strings_740,
    'Vmp': 41.58,
    'Voc': 49.97,
    'Imp': 17.8,
    'Isc': 18.77
}

inv = {
    'Max Input Voltage': 1500,
    'No. of MPPTs': 6,
    'Max Current per MPPT': 106,
    'Max Short Circuit per MPPT': 180,
    'Max PV Inputs per MPPT': 6
}

inverters = 464
PSTs = 29
inverters_per_PST = inverters / PSTs
if inverters_per_PST % 1 != 0:
    inverter_per_PST_up = math.ceil(inverters_per_PST)
    inverter_per_PST_down = math.floor(inverters_per_PST)
    print('Inverters per PST is not a multiple of 1.')
    print(f'The number of inverters per PSTs might be: {inverter_per_PST_up} and {inverter_per_PST_down}')
else:
    print(f'The number of inverters per PST might be: {inverters_per_PST}')

The number of inverters per PST might be: 16.0


Strings per MPPT and per inverter
=
- Loops over the number of strings in parallel on one MPPT and checks the MPPT current limits with the **740 Wp (`mod2`)** module (the higher-current bin): `n · Isc ≤ Max Short Circuit per MPPT` and `n · Imp ≤ Max Current per MPPT`.
- `max_number_of_PV` = the smallest of the two current limits and `Max PV Inputs per MPPT`. Strings per inverter = that × `No. of MPPTs`.
  - Warning: `range(6)` stops at 5, so 6 strings is never tested, and each iteration overwrites the previous one, so only the last `n = 5` really counts. The result (5) is right here only because 6 × 17.8 A is above 106 A anyway.
- `strings_per_inverter` = total strings / inverters. If it is not an integer, the split is between `ceil` and `ceil − 1` strings per inverter.
- The f-string with `inv['No. of MPPTs']` inside single quotes needs Python 3.12+.

In [3]:
for i in range(inv['Max PV Inputs per MPPT']):
    sc_current = i*mod2['Isc']
    mp_current = i*mod2['Imp']
    max_number_of_PV_ISC = i
    max_number_of_PV_MP = i

    if sc_current > inv['Max Short Circuit per MPPT']:
        sc_current = (i-1)*mod2['Isc']
        max_number_of_PV_ISC = (i-1)
    if mp_current > inv['Max Current per MPPT']:
        mp_current = (i-1)*mod2['Imp']
        max_number_of_PV_MP = (i-1)

max_number_of_PV = min(max_number_of_PV_MP, max_number_of_PV_ISC, inv['Max PV Inputs per MPPT'])

print(f'The max number of strings per MPPT is: {max_number_of_PV}')
print(f'The max number of strings per inverter is: {max_number_of_PV*inv['No. of MPPTs']}')

strings_per_inverter = (mod1['Strings']+mod2['Strings'])/inverters
strings_per_inverter_up = math.ceil(strings_per_inverter)
if strings_per_inverter_up != strings_per_inverter:
    print('\nStrings per inverter is not a multiple of 1.')
    print(f'The number of strings per inverter might be: {strings_per_inverter_up} and {strings_per_inverter_up-1}')
else:
    print(f'The number of strings per inverter might be: {strings_per_inverter}')

The max number of strings per MPPT is: 5
The max number of strings per inverter is: 30

Strings per inverter is not a multiple of 1.
The number of strings per inverter might be: 27 and 26


Max modules per string
=
- Increases the number of modules until `n · Voc` goes over `Max Input Voltage`, and keeps the previous value (30 with the 740 Wp Voc). It uses Voc at STC, with no low-temperature correction: 30 × 49.97 V = 1499.1 V leaves almost no margin.
- The result is then overwritten with the hardcoded `modules_per_inverter = 30` (the name means modules **per string**), so the loop only confirms it.

In [4]:
i = 1
while True:
    string_voltage = i * mod2['Voc']
    if string_voltage > inv['Max Input Voltage']:
        string_voltage = (i-1)*mod2['Voc']
        modules_per_inverter = i-1
        break
    i += 1

modules_per_inverter = 30
print(f'The max number of modules per string is: {modules_per_inverter}')

The max number of modules per string is: 30


26 / 27 strings per inverter, per module bin
=
Brute-force search for how many inverters of each bin get 26 or 27 strings.
- 735 Wp: the total is fixed at `ceil(9671 / strings_per_inverter) − 1` inverters, and each split `i` (26-string) / `total − i` (27-string) is checked for strings (`condition1`) and modules (`condition2`). The module check is redundant, because `Qtd.` is `Strings × 30`.
- 740 Wp: warning, `max_number_of_740_inverters = 1500` is hardcoded (the formula is commented out). Every split then has 1500 inverters, which can never give 2418 strings (26 × 1500 = 39000), so `condition3` is never true and the cell prints nothing (no saved output).
- Prints the four counts only if all four conditions are true. `print(i)` is a leftover debug print.

In [5]:
max_number_of_735_inverters = math.ceil(mod1['Strings']/strings_per_inverter)
for i in range(max_number_of_735_inverters):
    inv_26_735 = i
    inv_27_735 = max_number_of_735_inverters - i-1
    condition1 = inv_26_735*26 + inv_27_735*27 == mod1['Strings']
    condition2 = inv_26_735*26*modules_per_inverter + inv_27_735*27*modules_per_inverter == mod1['Qtd.']
    if condition1 and condition2:
        break

max_number_of_740_inverters = 1500 #math.ceil(mod2['Strings']/strings_per_inverter)
for i in range(max_number_of_740_inverters):
    inv_26_740 = i
    inv_27_740 = max_number_of_740_inverters - i
    condition3 = inv_26_740*26 + inv_27_740*27 == mod2['Strings']
    condition4 = inv_26_740*26*modules_per_inverter + inv_27_740*27*modules_per_inverter == mod2['Qtd.']
    if condition3 and condition4:
        print(i)
        break

if condition1 and condition2 and condition3 and condition4:
    print(f'The number of inverters with modules of 735Wp with 26 strings per inverter is: {inv_26_735}')
    print(f'The number of inverters with modules of 735Wp with 27 strings per inverter is: {inv_27_735}')
    print(f'The number of inverters with modules of 740Wp with 26 strings per inverter is: {inv_26_740}')
    print(f'The number of inverters with modules of 740Wp with 27 strings per inverter is: {inv_27_740}')

Helper: 26- and 27-string combinations for a target
=
`find_complete_strings(target)` looks for `i` inverters with 26 strings and `j` with 27 that add up to `target` strings (`target = 152`, hardcoded).
- Exact matches: returns a list of every `(i, j, i + j, target)`.
- No exact match: searches outward (`target + 1`, `target − 1`, `target + 2`, ...) and returns the first `(i, j, value)` found, as a single tuple with no `i + j`. The higher value is tried first.
- The saved output `(6, 0, 156)` is a fallback result: 152 has no exact split, and 156 = 6 × 26 is the nearest.
- The commented prints still refer to ×28 / ×29. `import math` is repeated from the first cell.

In [6]:
import math

target = 152


def find_complete_strings(target):
    result = []
    # Check exact match first
    for i in range(math.ceil(target/26) + 1):
        for j in range(math.ceil(target/27) + 1):
            if i*26 + j*27 == target:
                result.append((i,j,i+j,target))
    if result:
        return result

    # If no exact match, search outward
    offset = 1
    while True:
        # Check higher
        for i in range(math.ceil((target+offset)/26) + 1):
            for j in range(math.ceil((target+offset)/27) + 1):
                if i*26 + j*27 == target + offset:
                    return i, j, target + offset

        # Check lower (if positive)
        if target - offset >= 0:
            for i in range(math.ceil((target-offset)/26) + 1):
                for j in range(math.ceil((target-offset)/27) + 1):
                    if i*26 + j*27 == target - offset:
                        return i, j, target - offset

        offset += 1

result = find_complete_strings(target)
#print(f"The closest number to {target} that works is: {result[2]}")
#print(f"Using {result[0]}×28 + {result[1]}×29 = {result[2]}")
#print(f'The difference between {target} and {result[2]} is {abs(target - result[2])}')
print(result)

(6, 0, 156)


Helper: 25-, 26- and 27-string combinations for a target
=
Same search with three inverter sizes (25, 26 and 27 strings), for `target = 651`.
- Exact matches: returns a list of labelled tuples (`25:k`, `26:i`, `27:j`, `Inversores:k+i+j`, `Target:...`) and prints them all.
- No exact match: returns the first combination found outward as a single string.
  - Warning: in that case the loops iterate over the characters of that string, and `i[3]` raises `IndexError`.
- `sul_antes`, `norte_antes`, `sul_atual`, `norte_atual` and `tudo` are not used. Also note `sul_atual + norte_atual` = 12099, not 12089.
- The `continue` in the print loop does nothing, and the second loop (filter on `Inversores:272`) has no effect.

In [7]:
sul_antes = 6998
norte_antes = 5091

sul_atual = 6928
norte_atual = 5171


tudo = 12089
target = 651

def find_complete_strings(target):
    result = []
    # Check exact match first
    for k in range(math.ceil(target/25) + 1):
        for i in range(math.ceil(target/26) + 1):
            for j in range(math.ceil(target/27) + 1):
                if k*25 + i*26 + j*27 == target:
                    result.append((f"25:{k}",f"26:{i}",f"27:{j}",f"Inversores:{k+i+j}",f"Target:{target}"))
    if result:
        return result

    # If no exact match, search outward
    offset = 1
    while True:
        # Check higher
        for k in range(math.ceil((target+offset)/25) + 1):
            for i in range(math.ceil((target+offset)/26) + 1):
                for j in range(math.ceil((target+offset)/27) + 1):
                    if k*25 + i*26 + j*27 == target + offset:
                        return f"25:{k}, 26:{i}, 27:{j}, alvo:{target + offset}"

        # Check lower (if positive)
        if target - offset >= 0:
            for k in range(math.ceil((target-offset)/25) + 1):
                for i in range(math.ceil((target-offset)/26) + 1):
                    for j in range(math.ceil((target-offset)/27) + 1):
                        if k*25 + i*26 + j*27 == target - offset:
                            return f"25:{k}, 26:{i}, 27:{j}, alvo:{target - offset}"

        offset += 1

result = find_complete_strings(target)
#print(f"The closest number to {target} that works is: {result[2]}")
#print(f"Using {result[0]}×28 + {result[1]}×29 = {result[2]}")
#print(f'The difference between {target} and {result[2]} is {abs(target - result[2])}')
#print(result)
for i in result:
    print(i)
    continue

for i in result:
    if i[3] == 'Inversores:272':
        #print(i)
        continue

('25:0', '26:24', '27:1', 'Inversores:25', 'Target:651')
('25:1', '26:22', '27:2', 'Inversores:25', 'Target:651')
('25:2', '26:20', '27:3', 'Inversores:25', 'Target:651')
('25:3', '26:18', '27:4', 'Inversores:25', 'Target:651')
('25:4', '26:16', '27:5', 'Inversores:25', 'Target:651')
('25:5', '26:14', '27:6', 'Inversores:25', 'Target:651')
('25:6', '26:12', '27:7', 'Inversores:25', 'Target:651')
('25:7', '26:10', '27:8', 'Inversores:25', 'Target:651')
('25:8', '26:8', '27:9', 'Inversores:25', 'Target:651')
('25:9', '26:6', '27:10', 'Inversores:25', 'Target:651')
('25:10', '26:4', '27:11', 'Inversores:25', 'Target:651')
('25:11', '26:2', '27:12', 'Inversores:25', 'Target:651')
('25:12', '26:0', '27:13', 'Inversores:25', 'Target:651')
('25:25', '26:1', '27:0', 'Inversores:26', 'Target:651')
